# rackmains / ups / battery

**Intent**
- Gives `rackmains_guard_{shed_battery,restore_mains}_seconds`, the shutdown guard's required thresholds, as constants, and must:
  - Give them from its first release, through the project path every later model uses.
  - Carry optional outputs the fitted model will fill.
  - Fill `rackmains_guard_shed_runtime_seconds`, `rackmains_battery_runtime_remaining_seconds` and `rackmains_battery_energy_remaining_pct` once it is fitted.
  - Carry `rackmains_battery_energy_used_wh`, `rackmains_model_refit_due`, `rackmains_model_refit_reason_code` and `rackmains_battery_state_out`, with the state carried by the caller.
  - Accept optional inputs a fitted or stateful model will use, so that refit changes only the model file.
  - Treat an omitted optional input, which a wasm module receives as NaN, as its stated default; the initial state is all zeros.

**Data**
- `golden` is the fixed samples and expected outputs, and is all there is.
- No data is collected yet, so there is no `training`, `validation` or `backtest`.
- They follow with the fit, from the outage CSV files and the plug pairs, and the first drain to the knee, trial 5.
- Each outage CSV carries every input under its signature name, with `time` as the reading time, so each is a run.
- The label of a drain is the last on-battery row of a file closed `interrupted`.

**Status**
- The constants project is built and is the first release, and embedded in rackmains, which refuses to start without it.
- The signature was changed in place before the first rackmains release, under the embedded target exception, with the `rackmains_` prefix on every name.
- No data is collected yet, so no fitted or stateful model is built until a drain to the knee exists.

**Future**
- Overall: the design holds across every phase:
  - Keep the six state slots, so a stateful model is a refit changing only the model file.
  - Let the caller carry `rackmains_battery_state_in` and `rackmains_battery_state_out` unchanged, and treat a lost state as a battery of unknown history.
  - Track battery replacement and a six-monthly look at trial 1 outside the model, as a Home Assistant reminder.
- Phase 1: now, while the constants ship:
  - Run the guard on the constants in dry-run, and record every outage as a CSV row per poll.
  - Collect the outage CSV files and the plug pairs, which give a load to watts map.
  - Collect the first drain to the knee, trial 5, which is the data the fit needs.
  - Write golden rows from at least three sequences worked by hand from the state slots.
  - Replay each outage file from its `start` row state, one call per row, with `rackmains_read_failures` as recorded.
- Phase 2: once a drain to the knee exists:
  - Build the stateful model as `wasm`, replaying the recorded outages from their `start` row state to prove it.
  - Find the time to the knee, and fit `usable_wh` from the load through the drain, shaded to the safe side.
  - Fill the shed runtime and the remaining runtime and energy outputs from the measured shed time plus a margin.
  - Backtest every outage row by row, so the estimate never exceeds the real remaining time by a stated margin.
  - Define where the guard persists the carried state, with the project hash it came from.
- Phase 3: ongoing, as outages and batteries accumulate:
  - Raise `rackmains_model_refit_due` when the fit is old, the battery is replaced or observed drains stop matching predictions.
  - Refit on each new battery, and release each refit as a change to the model file only.


## Setup ([6.7.1](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#671-setup))

In [ ]:
from mlmodels import setup_workspace

ws = setup_workspace("rackmains/ups/battery")

## Collect ([6.7.2](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#672-collect))

In [ ]:
def golden_rows():
    clock = "clock_epoch_seconds"
    load = "rackmains_output_load_pct"
    failed = "rackmains_mains_failed"
    calibrating = "rackmains_ups_calibrating"
    low = "rackmains_battery_low"
    volts = "rackmains_battery_volts"
    replace = "rackmains_battery_replace_due"
    age = "rackmains_battery_age_days"
    read_failures = "rackmains_read_failures"
    state = "rackmains_battery_state_in"
    return {
        "case": [
            "first call with no state",
            "mid-drain with state",
            "replace flag raised",
            "required input missing",
            "reads failing, last load held",
            "every optional input given",
        ],
        "expect": ["ok", "ok", "ok", "error", "ok", "ok"],
        clock: [1793355300.0, 1793355330.0, 1793355360.0, 1793355390.0, 1793355420.0, 1793355450.0],
        load: [13, 13, 12, None, 12, 14],
        failed: [1, 1, 0, 1, 1, 1],
        calibrating: [None, None, None, None, None, 0],
        low: [None, None, None, None, None, 1],
        volts: [None, None, None, None, None, 11.6],
        replace: [0, 0, 1, 0, None, 0],
        age: [None, None, None, None, None, 730],
        read_failures: [None, None, None, None, 3, 0],
        state: [
            None,
            "0.0 1235.0 0.0 1.0 1.0 1793355329.0",
            "0.0 0.0 600.0 0.0 1.0 1793355359.0",
            None,
            "0.0 1500.0 0.0 1.0 1.0 1793355389.0",
            "0.0 1730.0 0.0 1.0 1.0 1793355419.0",
        ],
    }


golden = golden_rows()

## Define ([6.7.6](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#676-define))

In [ ]:
from mlmodels import define_model

CONSTANTS = {
    "rackmains_guard_shed_battery_seconds": 120,
    "rackmains_guard_restore_mains_seconds": 300,
}

define_model(ws, model=1000, flavour="static", source=CONSTANTS, label="A conservative starting set of guard times, not fitted")

## Package ([6.7.8](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#678-package))

In [ ]:
from mlmodels import package_model

golden = golden_rows()
package_model(ws, 1000, golden, target={"kind": "embedded", "language": "rust"})